In [1]:
# =========================
# LOAD LIBRARY
# =========================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
import pickle
from xgboost import XGBClassifier

from sklearn.preprocessing import LabelEncoder

In [2]:
# =========================
# LOAD DATA
# =========================

try:

    df = pd.read_csv("Data_set/updated.csv")
    print("File loaded successfully")
    
except FileNotFoundError as e:
    print(f"File not found: {e}.")
    
except Exception as e1:
    print(f"Something went wrong: {e1}.")

File loaded successfully


In [3]:
print(f"Les see the Old data Into this CSV file they have total {df.shape[0]} rows and the {df.shape[1]} columns.")

Les see the Old data Into this CSV file they have total 5572 rows and the 3 columns.


In [4]:
df = df.drop(['Unnamed: 0'],axis=1)

In [5]:
df.head()

,input,output
0,"go until jurong point, crazy available only in...",human
1,ok lar joking wif u oni,human
2,free entry in a wkly comp to win fa cup fina...,spam
3,u dun say so early hor u c already then say,human
4,"nah i don't think he goes to usf, he lives aro...",human


In [6]:
df['input'] = df['input'].fillna(df['input'].mode()[0])
print("EDA Completed.")

EDA Completed.


In [7]:
# =========================
# DATA BALANCING 
# =========================

df_human = df[df['output'] == 'human']
df_spam = df[df['output'] == 'spam']

new_spam_upscale = resample(
    df_spam,
    replace=True,
    n_samples = len(df_human),
    random_state=42
)

new_df = pd.concat([df_human,new_spam_upscale])

print("Data Balanced.")

print(new_df['output'].value_counts())

Data Balanced.
output
human    4825
spam     4825
Name: count, dtype: int64


In [8]:
x=new_df['input']
y=new_df['output']

x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.2)
print("Data Training testing is completed.")

Data Training testing is completed.


In [9]:
#Using Vectorized

vectorizer = TfidfVectorizer()

x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

print("Vectorization is completed.")

Vectorization is completed.


In [10]:
#MultinomialNB Model Training

lnb_model = MultinomialNB()

lnb_model.fit(x_train_tfidf,y_train)

lnb_prediciotn = lnb_model.predict(x_test_tfidf)

print("MultinomialNB Model trained successfully.")

MultinomialNB Model trained successfully.


In [11]:
# Lets see the mathamatics terms
# For MultinomialNB Model
print("Lets see the mathamatics terms For MultinomialNB Model.\n")

MultinomialNB_accuracy = round(accuracy_score(y_test,lnb_prediciotn)*100,2)

print("Accuracy Score of MultinomialNB model:",MultinomialNB_accuracy)
print("\nClassification Report:", classification_report(y_test,lnb_prediciotn))
print("\nconfusion matrix:\n", confusion_matrix(y_test,lnb_prediciotn))

Lets see the mathamatics terms For MultinomialNB Model.

Accuracy Score of MultinomialNB model: 98.29

Classification Report:               precision    recall  f1-score   support

       human       0.98      0.99      0.98       985
        spam       0.99      0.98      0.98       945

    accuracy                           0.98      1930
   macro avg       0.98      0.98      0.98      1930
weighted avg       0.98      0.98      0.98      1930


confusion matrix:
 [[971  14]
 [ 19 926]]


In [12]:
# Logistic REgression model Training

lr_model = LogisticRegression()

lr_model.fit(x_train_tfidf,y_train)

lr_prediciton = lr_model.predict(x_test_tfidf)

print("Logistic Regression Model trained successfully.")


Logistic Regression Model trained successfully.


In [13]:

# Lets see the mathamatics terms
# Logistic Regression Model

print("Lets see the mathamatics terms For Logistic Regression.\n")

logistic_regressor_accuracy = round(accuracy_score(y_test,lr_prediciton)*100,2)

print("Accuracy Score of the Logistic Regressio odel:",logistic_regressor_accuracy)
print("\nClassification Report:", classification_report(y_test,lr_prediciton))
print("\nconfusion matrix:\n", confusion_matrix(y_test,lr_prediciton))



Lets see the mathamatics terms For Logistic Regression.

Accuracy Score of the Logistic Regressio odel: 98.5

Classification Report:               precision    recall  f1-score   support

       human       0.98      0.99      0.99       985
        spam       0.99      0.98      0.98       945

    accuracy                           0.98      1930
   macro avg       0.99      0.98      0.98      1930
weighted avg       0.99      0.98      0.98      1930


confusion matrix:
 [[978   7]
 [ 22 923]]


In [14]:
# encode labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# model
xgb = XGBClassifier(
    random_state=42,
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6
)

# train
xgb.fit(x_train_tfidf, y_train_encoded)

# predict
xgb_prediction = xgb.predict(x_test_tfidf)

# decode back (optional)
xgb_prediction_labels = le.inverse_transform(xgb_prediction)

# accuracy
xgb_accuracy = round(accuracy_score(y_test, xgb_prediction_labels)*100,2)

print("Accuracy Score:", xgb_accuracy)
print("\nClassification Report:\n", classification_report(y_test, xgb_prediction_labels))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, xgb_prediction_labels))

Accuracy Score: 98.96

Classification Report:
               precision    recall  f1-score   support

       human       0.99      0.99      0.99       985
        spam       0.99      0.99      0.99       945

    accuracy                           0.99      1930
   macro avg       0.99      0.99      0.99      1930
weighted avg       0.99      0.99      0.99      1930


Confusion Matrix:
 [[974  11]
 [  9 936]]


In [15]:
# Compare the accuracy of all models

if MultinomialNB_accuracy > logistic_regressor_accuracy and MultinomialNB_accuracy > xgb_accuracy:
    print(f"MultinomialNB has highest accuracy: {MultinomialNB_accuracy}")

elif logistic_regressor_accuracy > xgb_accuracy:
    print(f"Logistic Regression has highest accuracy: {logistic_regressor_accuracy}")

else:
    print(f"XGBoost has highest accuracy: {xgb_accuracy}")

XGBoost has highest accuracy: 98.96


In [24]:
#User Inout 

text =  input("Enter the Text:- ")

new_data = pd.DataFrame([{
    "text" : text
}])

unwanted_charecters = r'[.><+\-*/=!@#$%^&()\-:;?`~]'

new_data['text'] = new_data['text'].str.lower()
new_data['text'] = new_data['text'].replace(unwanted_charecters," ",regex=True)
new_data['text'] = new_data['text'].str.strip()
new_data['text'] = new_data['text'].replace(r'\s+',' ',regex=True)
new_data['text'] = new_data['text'].replace(r'\d+',' ',regex=True)


txt_output = vectorizer.transform(new_data['text'])


lnb_final_output = lnb_model.predict(txt_output)[0]
lr_final_output = lr_model.predict(txt_output)[0]

xgboost_prediction = xgb.predict(txt_output)
xgboost_prediction_output = le.inverse_transform(xgboost_prediction)[0]


print("\nModel Comparison Result:\n")

print(f"{'Model':<30} | {'Prediction':<10} | Accuracy")
print("-"*60)

print(f"{'Logistic Regression':<30} | {lr_final_output:<10} | {round(logistic_regressor_accuracy,2)}%")

print(f"{'Multinomial Naive Bayes':<30} | {lnb_final_output:<10} | {round(MultinomialNB_accuracy,2)}%")

print(f"{'XGboost Regression':<30} | {xgboost_prediction_output:<10} | {round(xgb_accuracy,2)}%")

print("-"*60)


Model Comparison Result:

Model                          | Prediction | Accuracy
------------------------------------------------------------
Logistic Regression            | human      | 98.5%
Multinomial Naive Bayes        | human      | 98.29%
XGboost Regression             | human      | 98.96%
------------------------------------------------------------


In [ ]:
#Make the pickle model
model = {
    "vectorizer_model" : vectorizer,
    "label_encoder":le,
    "MultinomialNB_model" : lnb_model,
    "Linear_regressor_model" : lr_model,
    "xgbooster_model":xgb,
    "lr_accuracy":logistic_regressor_accuracy,
    "lnb_accuracy":MultinomialNB_accuracy,
    "xgb_accuracy":xgb_accuracy,
}

try:
    with open("Models/pkl_model.pkl",'wb') as f:
        pickle.dump(model,f)
except Exception as e3:
    print(f"Error {e3}.")
    
print("Pickle model is Created.")

Pickle model is Created.
